In [26]:
import xarray as xr
#manually loaded and transformed NC files to .tif rasters
ds = xr.open_dataset(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\C3S-LC-L4-LCCS-Map-300m-P1Y-2020-v2.1.1.area-subset.5.36.-5.26.nc")

print(ds)

<xarray.Dataset> Size: 156MB
Dimensions:              (time: 1, lat: 3600, lon: 3600, bounds: 2)
Coordinates:
  * time                 (time) datetime64[ns] 8B 2020-01-01
  * lat                  (lat) float64 29kB 4.999 4.996 4.993 ... -4.996 -4.999
  * lon                  (lon) float64 29kB 26.0 26.0 26.01 ... 35.99 36.0 36.0
Dimensions without coordinates: bounds
Data variables:
    lccs_class           (time, lat, lon) uint8 13MB ...
    processed_flag       (time, lat, lon) float32 52MB ...
    current_pixel_state  (time, lat, lon) float32 52MB ...
    observation_count    (time, lat, lon) uint16 26MB ...
    change_count         (time, lat, lon) uint8 13MB ...
    crs                  int32 4B ...
    lat_bounds           (lat, bounds) float64 58kB ...
    lon_bounds           (lon, bounds) float64 58kB ...
    time_bounds          (time, bounds) datetime64[ns] 16B ...
Attributes: (12/38)
    title:                      Land Cover Map of 2020
    summary:                    This

In [6]:
import rioxarray


In [27]:
lc = ds["lccs_class"].isel(time=0)

lc.rio.write_crs("EPSG:4326", inplace=True)

lc.rio.to_raster(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\2020.tif")

In [29]:
# clip global rasters to square covering Uganda with margin
import rasterio
from rasterio.windows import from_bounds
from pathlib import Path

in_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)")
out_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")
out_folder.mkdir(parents=True, exist_ok=True)

# target bounds
west, south, east, north = 26, -5, 36, 5

for year in range(2006, 2016):
    in_raster = in_folder / f"{year}.tif"
    out_raster = out_folder / f"{year}.tif"

    print("Clipping", year)

    with rasterio.open(in_raster) as src:
        window = from_bounds(west, south, east, north, src.transform)

        data = src.read(1, window=window)

        transform = src.window_transform(window)

        meta = src.meta.copy()
        meta.update({
            "height": data.shape[0],
            "width": data.shape[1],
            "transform": transform,
            "compress": "lzw"
        })

        with rasterio.open(out_raster, "w", **meta) as dst:
            dst.write(data, 1)

Clipping 2006
Clipping 2007
Clipping 2008
Clipping 2009
Clipping 2010
Clipping 2011
Clipping 2012
Clipping 2013
Clipping 2014
Clipping 2015


In [31]:
#CSB
import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\qgis layers\30km_circles.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CSB.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CSB.xlsx",
    index=False
)

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020


In [1]:
#CANDIDATES
import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\CANDIDATES_30km_BUFFERS.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CANDIDATES60.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CANDIDATES60.xlsx",
    index=False
)

Processing 2001


C:\Users\Carl\anaconda3\envs\CSB_project\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009


KeyboardInterrupt: 

In [1]:
#hh panel
import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\5km_buffer_HH.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_hhpanel.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_hhpanel.xlsx",
    index=False
)

Processing 2001


C:\Users\Carl\anaconda3\envs\CSB_project\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020


In [6]:
# Rings 30-40

import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\30-40km ringbuffers.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_30-40km.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_30-40km.xlsx",
    index=False
)

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020


In [7]:
# Rings 40-50

import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\40-50km ringbuffers.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_40-50km.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_40-50km.xlsx",
    index=False
)

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020


In [5]:
# Rings 50-60

import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\50-60km ringbuffers.shp"
raster_folder = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

for raster in raster_folder.glob("*.tif"):

    year = raster.stem

    print("Processing", year)

    with rasterio.open(raster) as src:
        arr = src.read(1)
        nodata = src.nodata

    total = zonal_stats(
        buffers,
        str(raster),
        stats="count",
        nodata=nodata
    )

    total_pixels = np.array([x["count"] for x in total])

    for cls in classes:

        zs = zonal_stats(
            buffers,
            str(raster),
            categorical=True,
            nodata=nodata
        )

        cls_count = np.array([
            z.get(cls, 0) for z in zs
        ])

        share = cls_count / total_pixels

        buffers[f"lc{cls}_{year}"] = share


buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_50-60km.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_50-60km.xlsx",
    index=False
)

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020


In [2]:
# CANDIDATES - land cover shares for 2010 only

import geopandas as gpd
import rasterio
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

buffer_path = r"C:\Users\Carl\Desktop\GIS\CANDIDATES_30km_BUFFERS.shp"
raster_path = Path(r"C:\Users\Carl\Desktop\SACCI-LC-L4-LCCS-Map-300m P1Y (2006-2015)\clipped\2010.tif")

buffers = gpd.read_file(buffer_path)

buffers = buffers[buffers.geometry.notna()].copy()
buffers["geometry"] = buffers.geometry.make_valid()
buffers = buffers[~buffers.geometry.is_empty].copy()

buffers = buffers.to_crs("EPSG:4326")

classes = [10, 20, 30, 40]

year = raster_path.stem

print("Processing", year)

with rasterio.open(raster_path) as src:
    nodata = src.nodata

total = zonal_stats(
    buffers,
    str(raster_path),
    stats="count",
    nodata=nodata
)

total_pixels = np.array([x["count"] for x in total])

zs = zonal_stats(
    buffers,
    str(raster_path),
    categorical=True,
    nodata=nodata
)

for cls in classes:
    cls_count = np.array([
        z.get(cls, 0) for z in zs
    ])

    share = cls_count / total_pixels

    buffers[f"lc{cls}_{year}"] = share

buffers.to_file(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CANDIDATES60_2010.gpkg",
    driver="GPKG"
)

buffers.drop(columns="geometry").to_excel(
    r"C:\Users\Carl\Desktop\landcover_output\lccs_CANDIDATES60_2010.xlsx",
    index=False
)

Processing 2010
